In [1]:
pip install earthengine-api requests

  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.0 MB ? eta -:--:--
    --------------------------------------- 0.3/15.0 MB ? eta -:--:--
    ---------------

In [6]:
import os
import ee
import geopandas as gpd
import requests
import zipfile
import io

# 1. Initialize Earth Engine
# This will open a web browser to authenticate your Google Account the first time you run it.
try:
    ee.Initialize(project='ee-ayushsingh2005811')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-ayushsingh2005811')
# 2. Define Paths and Output Directory
aoi_file = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\aoi\aoi.geojson"
output_dir = 'data/raw/satellite'
os.makedirs(output_dir, exist_ok=True)

# 3. Load AOI and convert to Earth Engine Geometry
aoi_gdf = gpd.read_file(aoi_file)
coords = aoi_gdf.geometry.iloc[0].__geo_interface__['coordinates']
aoi_ee = ee.Geometry.Polygon(coords)

# 4. Set Parameters
start_date = '2023-04-25'
end_date = '2026-04-25'
cloud_threshold = 10
bands_to_download = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

# 5. Query Sentinel-2 Surface Reflectance
print("Querying Earth Engine for Sentinel-2 data...")
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi_ee)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold)))

# 6. Calculate the median composite and clip to AOI
median_image = s2_collection.median().clip(aoi_ee)

# 7. Download and Save Loop
for band in bands_to_download:
    out_path = os.path.join(output_dir, f'{band}.tif')
    
    print(f"Requesting {band}...")
    
    # Isolate the single band
    single_band_img = median_image.select(band)
    
    # Generate the download URL
    # We enforce a scale of 10m so all bands perfectly align (B11 and B12 are natively 20m)
    url = single_band_img.getDownloadURL({
        'scale': 10,
        'crs': 'EPSG:4326',
        'region': aoi_ee,
        'format': 'GEO_TIFF'
    })
    
    # Download the zipped data
    response = requests.get(url)

    if response.status_code == 200:
        with open(out_path, 'wb') as f:
            f.write(response.content)
    else:
        print(f"Failed for {band}: {response.text}")            
        print(f"Saved: {out_path}")

print("\nCheckpoint: Raw spectral bands collected and perfectly aligned.")

Querying Earth Engine for Sentinel-2 data...
Requesting B2...
Requesting B3...
Requesting B4...
Requesting B8...
Requesting B11...
Requesting B12...

Checkpoint: Raw spectral bands collected and perfectly aligned.


In [7]:
import os
import rasterio
import numpy as np

# 1. Define Absolute Paths
input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\satellite"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\indices"

os.makedirs(output_dir, exist_ok=True)

# 2. Helper function to read a band and convert to true reflectance (0-1)
def read_and_scale_band(band_name):
    path = os.path.join(input_dir, f'{band_name}.tif')
    with rasterio.open(path) as src:
        # Read as float32 to prevent integer truncation during division
        array = src.read(1).astype('float32')
        # Sentinel-2 scale factor is 10000
        array = array / 10000.0 
        meta = src.meta.copy()
    return array, meta

print("Loading and scaling bands...")
blue, meta = read_and_scale_band('B2')
red, _ = read_and_scale_band('B4')
nir, _ = read_and_scale_band('B8')
swir1, _ = read_and_scale_band('B11')

# Ensure output rasters use float32 to store decimal index values safely
meta.update(dtype=rasterio.float32)

# Ignore runtime warnings for zero division (common at raster edges or water bodies)
np.seterr(divide='ignore', invalid='ignore')

# 3. Calculate Indices

print("Calculating NDVI...")
# NDVI = (NIR - Red) / (NIR + Red)
ndvi = (nir - red) / (nir + red)

print("Calculating EVI...")
# EVI = 2.5 * ((NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1))
evi = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))

print("Calculating SAVI...")
# SAVI = ((NIR - Red) / (NIR + Red + 0.5)) * 1.5
savi = ((nir - red) / (nir + red + 0.5)) * 1.5

print("Calculating BSI...")
# BSI = ((SWIR1 + Red) - (NIR + Blue)) / ((SWIR1 + Red) + (NIR + Blue))
bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue))

print("Calculating NDMI...")
# NDMI = (NIR - SWIR1) / (NIR + SWIR1)
ndmi = (nir - swir1) / (nir + swir1)

# 4. Helper function to save index rasters safely
def save_index(array, filename):
    # Convert infinite values or NaNs (from zero division) to a NoData value like -9999
    array = np.nan_to_num(array, nan=-9999.0, posinf=-9999.0, neginf=-9999.0)
    meta.update(nodata=-9999.0)
    
    out_path = os.path.join(output_dir, filename)
    with rasterio.open(out_path, 'w', **meta) as dest:
        dest.write(array, 1)
    print(f"Saved: {out_path}")

# 5. Export all layers
print("\nExporting layers...")
save_index(ndvi, 'ndvi.tif')
save_index(evi, 'evi.tif')
save_index(savi, 'savi.tif')
save_index(bsi, 'bsi.tif')
save_index(ndmi, 'ndmi.tif')

print("\nCheckpoint: Vegetation layers ready.")

Loading and scaling bands...
Calculating NDVI...
Calculating EVI...
Calculating SAVI...
Calculating BSI...
Calculating NDMI...

Exporting layers...
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\indices\ndvi.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\indices\evi.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\indices\savi.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\indices\bsi.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\indices\ndmi.tif

Checkpoint: Vegetation layers ready.


In [9]:
import os
import ee
import geopandas as gpd
import requests
import zipfile
import io

try:
    ee.Initialize(project='ee-ayushsingh2005811')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-ayushsingh2005811')

# 2. Define Absolute Paths
aoi_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\aoi\aoi.geojson"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem"

os.makedirs(output_dir, exist_ok=True)

# 3. Load AOI and convert to Earth Engine Geometry
aoi_gdf = gpd.read_file(aoi_path)
coords = aoi_gdf.geometry.iloc[0].__geo_interface__['coordinates']
aoi_ee = ee.Geometry.Polygon(coords)

# 4. Fetch NASA SRTM DEM
print("Querying NASA SRTM DEM...")
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi_ee)

# 5. Derive Terrain Layers (Slope and Aspect)
# ee.Terrain.products calculates slope (degrees) and aspect (degrees) automatically
terrain = ee.Terrain.products(dem)

# Define the bands we want to export and their corresponding filenames
layers_to_export = {
    'elevation': 'dem.tif',
    'slope': 'slope.tif',
    'aspect': 'aspect.tif'
}

# 6. Download and Save Loop
for band, filename in layers_to_export.items():
    out_path = os.path.join(output_dir, filename)
    print(f"Requesting {band}...")
    
    # Isolate the specific terrain band
    single_band_img = terrain.select(band)
    
    # Generate download URL (forcing 30m native SRTM resolution)
    url = single_band_img.getDownloadURL({
        'scale': 30,
        'crs': 'EPSG:4326',
        'region': aoi_ee,
        'format': 'GEO_TIFF'
    })
    
    # Download and extract the TIF
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(out_path, 'wb') as f:
            f.write(response.content)
    else:
        print(f"Failed for {band}: {response.text}")
            
    print(f"Saved: {out_path}")

print("\nCheckpoint: Terrain stack ready.")

Querying NASA SRTM DEM...
Requesting elevation...
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\dem.tif
Requesting slope...
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\slope.tif
Requesting aspect...
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\aspect.tif

Checkpoint: Terrain stack ready.


In [10]:
import rasterio

def check_raster(filepath):
    with rasterio.open(filepath) as src:
        array = src.read(1)
        print(f"File: {filepath.split('/')[-1]}")
        print(f"  Min value: {array.min()}")
        print(f"  Max value: {array.max()}")
        print(f"  Mean value: {array.mean():.2f}\n")

# Use your exact paths
check_raster(r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\dem.tif")
check_raster(r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\slope.tif")
check_raster(r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\aspect.tif")

File: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\dem.tif
  Min value: 851
  Max value: 890
  Mean value: 864.30

File: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\slope.tif
  Min value: 0
  Max value: 14
  Mean value: 2.01

File: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\dem\aspect.tif
  Min value: 0
  Max value: 353
  Mean value: 129.54



In [11]:
import os
import ee
import geopandas as gpd
import requests
import zipfile
import io

# 1. Initialize Earth Engine
try:
    ee.Initialize(project='ee-ayushsingh2005811')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-ayushsingh2005811')

# 2. Define Absolute Paths
aoi_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\aoi\aoi.geojson"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\landcover"

os.makedirs(output_dir, exist_ok=True)

# 3. Load AOI and convert to Earth Engine Geometry
aoi_gdf = gpd.read_file(aoi_path)
coords = aoi_gdf.geometry.iloc[0].__geo_interface__['coordinates']
aoi_ee = ee.Geometry.Polygon(coords)

# 4. Fetch ESA WorldCover (10m resolution)
print("Querying ESA WorldCover data...")
# The collection contains a global mosaic. We select the first (and only) image and clip it.
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi_ee)

# The land classification values are stored in the 'Map' band
landcover_map = worldcover.select('Map')

# 5. Download and Save
out_path = os.path.join(output_dir, 'worldcover.tif')
print("Requesting worldcover.tif...")

# We enforce a scale of 10m so it perfectly overlays your Sentinel-2 vegetation indices
url = landcover_map.getDownloadURL({
    'scale': 10,
    'crs': 'EPSG:4326',
    'region': aoi_ee,
    'format': 'GEO_TIFF'
})


response = requests.get(url)

if response.status_code == 200:
    with open(out_path, 'wb') as f:
        f.write(response.content)
else:
    print(f"Failed for {band}: {response.text}")
        
print(f"Saved: {out_path}")
print("\nCheckpoint: Landcover ready.")

Querying ESA WorldCover data...
Requesting worldcover.tif...
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\landcover\worldcover.tif

Checkpoint: Landcover ready.


In [12]:
import os
import pandas as pd

# 1. Define Paths
temp_fert_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\management\fertilizer.csv" # Update this if you saved it elsewhere
temp_land_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\management\irrigation.csv" # Update this if you saved it elsewhere
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\management"

os.makedirs(output_dir, exist_ok=True)

# 2. Process Fertilizer Data
try:
    df_fert = pd.read_csv(temp_fert_path)
    
    # Keep only the essential columns: Year, Item (Nutrient), and Value (Amount)
    df_fert_clean = df_fert[['Year', 'Item', 'Unit', 'Value']]
    
    # Rename columns for the pipeline
    df_fert_clean.columns = ['Year', 'Nutrient_Type', 'Unit', 'Amount']
    
    fert_out = os.path.join(output_dir, 'fertilizer.csv')
    df_fert_clean.to_csv(fert_out, index=False)
    print(f"Saved: {fert_out}")
    
except FileNotFoundError:
    print(f"Waiting for {temp_fert_path} to be downloaded...")

# 3. Process Irrigation/Land Data
try:
    df_land = pd.read_csv(temp_land_path)
    
    # Keep only the essential columns
    df_land_clean = df_land[['Year', 'Item', 'Unit', 'Value']]
    
    # Rename columns for the pipeline
    df_land_clean.columns = ['Year', 'Land_Category', 'Unit', 'Area']
    
    irrig_out = os.path.join(output_dir, 'irrigation.csv')
    df_land_clean.to_csv(irrig_out, index=False)
    print(f"Saved: {irrig_out}")
    
except FileNotFoundError:
    print(f"Waiting for {temp_land_path} to be downloaded...")

print("\nCheckpoint: Management data ready.")

Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\management\fertilizer.csv
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\management\irrigation.csv

Checkpoint: Management data ready.
